# P2C-64 v3.4 (`WACA_G2_V33` Fast Stage 3 ＋ 双方公平 Calibrated MPC ＋ Phase/Tension Dynamic Gating) — CPU 検証専用ノートブック

**GPU 不要・モデル再学習 0 分！** 既存の学習済みモデル（`run_egaroucid_25m_v33/{raw_ntuple64.p2c, waca_g2_v33.p2c}`）をビット完全一致のまま再利用し、**P2C-64 v3.4** の 3 つの純スカラー探索・推論最適化の効果を段階的アブレーションで厳密実証します。

1. **`WACA_G2_V33` Stage 3 の純スカラー大幅高速化**：448B `int32 h[14][8]` ＋ 128B 累積配列を 112B `int8 he[14]` のみへ圧縮し高 MLP ロード化、単一レジスタ完結 `(k, k+4)` ペアループでスタック退避と XMM を完全ゼロ化、`extract_waca_v33_stage3_extra_features`（4 隅空き早期ガード付き）＆ `eval_waca_v33_mlp_kernel`（4-way `i` 展開）の色非依存共有化により命令数を `-80%`（`3,454 → 687`）へ激減。
1. **`WACA_G2_V33` Stage 3 の純スカラー大幅高速化**：448B `h[14][8]` スタック配列を 32B 隅キャッシュ `hc[4][8]` ＋ オンザフライ処理へ削減、`(k, k+4)` ペアループで 16 GPR 完結、`extract_waca_v33_stage3_extra_features`（4 隅空き早期ガード付き）＆ `eval_waca_v33_mlp_kernel` の色非依存共有化により L1I 占有を激減。
2. **双方 100% 公平なモデル別・精密校正 MPC（Calibrated MPC）**：同一未見 2,000 局面から `RAW_NTUPLE_64` と `WACA_G2_V33` それぞれの 4 深さペア `(1,5)..(4,8)` × 6 手番バケット回帰 `(slope, intercept, sigma)` を自動校正し、60 手番区分線形補間 L1D テーブルで精密カット。
3. **フェーズ＆緊張度連動・動的ゲーティング（`--dyn-gate phase | phase_tension`）**：序盤はタイト（`0.85x`）、中終盤の勝負所（`ply 28..47`）は広め（`1.15x`）、さらに Stage 2.5 残差 `|r25|` の大きさに応じて Stage 3 窓を自動拡張（TT 健全性は固定 `tt_guard` で数学的保証）。

- **既存成果物を完全保護**：出力はすべて新規フォルダ `run_egaroucid_25m_v34/` へ保存します。

## Cell 1: Googleドライブマウント ＆ CPU環境・既存モデル確認（CPUランタイム専用）

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "run_egaroucid_25m_v34"
V20_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v20"
V33_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v33"
DRIVE_CACHE_DIR = "/content/drive/MyDrive/p2c64_data/cache_bin"
DRIVE_WORK_DIR = f"/content/drive/MyDrive/p2c64_runs/{RUN_NAME}"
REPO_DIR = "/content/p2c64-othello"

os.makedirs(DRIVE_WORK_DIR, exist_ok=True)

RAW_P2C = f"{V33_RUN_DIR}/raw_ntuple64.p2c"
if not os.path.exists(RAW_P2C):
    RAW_P2C = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
V33_P2C = f"{V33_RUN_DIR}/waca_g2_v33.p2c"
VAL_BIN = f"{DRIVE_CACHE_DIR}/val.bin"

for label, path in [("RAW_NTUPLE_64", RAW_P2C), ("WACA_G2_V33", V33_P2C), ("VAL_BIN", VAL_BIN)]:
    assert os.path.exists(path), f"Missing {label}: {path}"
    print(f"[OK] {label:<14}: {path} ({os.path.getsize(path) / (1024*1024):.2f} MiB)")

print(f"Output Dir     : {DRIVE_WORK_DIR}")
!lscpu | grep -E "Model name|CPU\(s\):|L1d|L1i|L2|L3" | head -n 12

## Cell 2: リポジトリ `clone` / `pull` ＆ 純スカラービルド・単体テスト・34カーネル 0-SIMD ASM監査・`bench_scalar_eval` 実測

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/Taichi111009/p2c64-othello.git"
REPO_DIR="/content/p2c64-othello"

if [ -d "$REPO_DIR/.git" ]; then
    cd "$REPO_DIR" && git fetch origin && git pull --ff-only origin main
else
    git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
make clean all test audit bench
ls -lh build/

## Cell 3: 双方 100% 公平なモデル別・精密 MPC 校正テーブル生成（同一未見 2,000 局面・4 深さペア × 6 手数バケット）

`raw_ntuple64.p2c` と `waca_g2_v33.p2c` の双方に対し、同一の未見 2,000 局面から `(1,5), (2,6), (3,7), (4,8)` × 6 バケットの回帰パラメータを自動計測し、`mpc_cal_raw.csv` と `mpc_cal_v33.csv` を生成・表示します。

In [ ]:
import os
import subprocess

MPC_BIN = f"{REPO_DIR}/build/measure_mpc_sigma"
CAL_RAW_CSV = f"{DRIVE_WORK_DIR}/mpc_cal_raw.csv"
CAL_V33_CSV = f"{DRIVE_WORK_DIR}/mpc_cal_v33.csv"

# Copy val.bin to local SSD for fast random-free reads
LOCAL_VAL = "/content/val.bin"
if not os.path.exists(LOCAL_VAL):
    import shutil
    shutil.copy2(VAL_BIN, LOCAL_VAL)

for label, model_p2c, out_csv in [
    ("RAW_NTUPLE_64", RAW_P2C, CAL_RAW_CSV),
    ("WACA_G2_V33",   V33_P2C, CAL_V33_CSV),
]:
    print(f"\n=== Calibrating MPC table for {label} (2,000 unseen pos) ===", flush=True)
    subprocess.run([
        MPC_BIN,
        "--data", LOCAL_VAL,
        "--num-pos", "2000",
        "--tt-mb", "64",
        "--gate-mode", "wide_safe_bnd",
        "--m25", "2.5",
        "--m3", "1.5",
        "--tt-guard", "5.5",
        "--calibrate-out", out_csv,
        "--models", model_p2c,
    ], check=True)

print("\n=== Generated mpc_cal_raw.csv ===")
!cat {CAL_RAW_CSV}
print("\n=== Generated mpc_cal_v33.csv ===")
!cat {CAL_V33_CSV}

## Cell 4A: 20ms 段階アブレーション検証（`b4_hybrid`, `wide_safe_bnd 2.5/1.5 g=5.5`, `seed=20261127`）

各要素の純粋な寄与を 1 段階ずつ切り分けて計測し、対局ごとに CSV 最終行を出力します：
1. **`RAW(calibrated) vs RAW(legacy)`**（600 局）：校正 MPC 単体の純効果確認
2. **`v3.3(fast S3 + legacy MPC + dyn=off) vs RAW(legacy)`**（1,000 局）：Stage 3 高速化単体の NPS・Elo 効果
3. **`v3.3(fast S3 + calibrated MPC + dyn=off) vs RAW(calibrated)`**（1,000 局）：双方公平・精密校正 MPC 導入効果
4. **`v3.3(fast S3 + calibrated MPC + dyn=phase) vs RAW(calibrated)`**（1,000 局）：フェーズ動的ゲート効果
5. **`v3.3(fast S3 + calibrated MPC + dyn=phase_tension) vs RAW(calibrated)`**（1,000 局）：緊張度連動動的ゲート効果

In [ ]:
import os
import subprocess

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
CAL_RAW_CSV = f"{DRIVE_WORK_DIR}/mpc_cal_raw.csv"
CAL_V33_CSV = f"{DRIVE_WORK_DIR}/mpc_cal_v33.csv"
CSV_4A = f"{DRIVE_WORK_DIR}/match_results_v34_ablation.csv"
TT_GUARD = "5.5"
DYN_R25_K = "0.50"

assert os.path.exists(CAL_RAW_CSV), f"Run Cell 3 first: missing {CAL_RAW_CSV}"
assert os.path.exists(CAL_V33_CSV), f"Run Cell 3 first: missing {CAL_V33_CSV}"

def run_match_and_log(label, cmd, csv_path):
    print(f"\n=== {label} ===", flush=True)
    subprocess.run(cmd, check=True)
    if os.path.exists(csv_path):
        with open(csv_path) as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if len(lines) >= 2:
            print(f"[CSV LAST ROW] {lines[-1]}", flush=True)

# 1) RAW(calibrated) vs RAW(legacy) (600g = 300 pairs)
run_match_and_log(
    "[4A-1] MPC Cal Ablation: RAW(calibrated) vs RAW(legacy) (20ms, b4_hybrid, 600g)",
    [
        MATCH_BIN, "--p1", RAW_P2C, "--p2", RAW_P2C,
        "--mode", "time", "--limit", "20", "--pairs", "300",
        "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
        "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
        "--mpc-cal-p1", CAL_RAW_CSV,
        "--mo", "b4_hybrid", "--seed", "20261127", "--csv", CSV_4A,
    ],
    CSV_4A,
)

# 2) v3.3(fast S3 + legacy MPC + dyn=off) vs RAW(legacy) (1,000g)
run_match_and_log(
    "[4A-2] Fast Stage 3 Only: v3.3(legacy MPC, dyn=off) vs RAW(legacy) (20ms, b4_hybrid, 1000g)",
    [
        MATCH_BIN, "--p1", V33_P2C, "--p2", RAW_P2C,
        "--mode", "time", "--limit", "20", "--pairs", "500",
        "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
        "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
        "--mo", "b4_hybrid", "--gate-mode", "wide_safe_bnd",
        "--m25", "2.5", "--m3", "1.5", "--tt-guard", TT_GUARD,
        "--dyn-gate", "off", "--seed", "20261127", "--csv", CSV_4A,
    ],
    CSV_4A,
)

# 3..5) Fair Calibrated MPC + DynGate Ablation (off / phase / phase_tension, 1,000g each)
for tag, dyn in [
    ("[4A-3] Fast S3 + Fair Calibrated MPC (dyn=off)",           "off"),
    ("[4A-4] Fast S3 + Fair Calibrated MPC (dyn=phase)",         "phase"),
    ("[4A-5] Fast S3 + Fair Calibrated MPC (dyn=phase_tension)", "phase_tension"),
]:
    run_match_and_log(
        f"{tag}: v3.3 vs RAW (20ms, b4_hybrid, 1000g)",
        [
            MATCH_BIN, "--p1", V33_P2C, "--p2", RAW_P2C,
            "--mode", "time", "--limit", "20", "--pairs", "500",
            "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
            "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
            "--mpc-cal-p1", CAL_V33_CSV, "--mpc-cal-p2", CAL_RAW_CSV,
            "--mo", "b4_hybrid", "--gate-mode", "wide_safe_bnd",
            "--m25", "2.5", "--m3", "1.5", "--tt-guard", TT_GUARD,
            "--dyn-gate", dyn, "--dyn-r25-k", DYN_R25_K,
            "--seed", "20261127", "--csv", CSV_4A,
        ],
        CSV_4A,
    )

print(f"\n=== [Cell 4A Done] Full Ablation CSV ({CSV_4A}) ===")
!cat {CSV_4A}

## Cell 4B: 50ms / 100ms 超深読み本決戦（`b4_hybrid` / `static` 双方・各 600 局）

Cell 4A で最良だった `DYN_GATE`（`off` / `phase` / `phase_tension`）および `USE_CAL_MPC` を冒頭変数で指定し、50ms（600局, `seed=20261127`）および 100ms（600局, `seed=20261128`）を `b4_hybrid` / `static` 双方で本検証して `match_results_v34_deep.csv` へ保存します。

In [ ]:
import os
import subprocess

# Select best settings from Cell 4A:
DYN_GATE = "phase_tension"  # "off" | "phase" | "phase_tension"
DYN_R25_K = "0.50"
USE_CAL_MPC = True
TT_GUARD = "5.5"

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
CAL_RAW_CSV = f"{DRIVE_WORK_DIR}/mpc_cal_raw.csv"
CAL_V33_CSV = f"{DRIVE_WORK_DIR}/mpc_cal_v33.csv"
CSV_4B = f"{DRIVE_WORK_DIR}/match_results_v34_deep.csv"

if USE_CAL_MPC:
    assert os.path.exists(CAL_RAW_CSV), f"Missing {CAL_RAW_CSV}"
    assert os.path.exists(CAL_V33_CSV), f"Missing {CAL_V33_CSV}"

def run_match_and_log(label, cmd, csv_path):
    print(f"\n=== {label} ===", flush=True)
    subprocess.run(cmd, check=True)
    if os.path.exists(csv_path):
        with open(csv_path) as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if len(lines) >= 2:
            print(f"[CSV LAST ROW] {lines[-1]}", flush=True)

suite_4b = [
    ("[4B-1] 50ms  B4_HYBRID (600g, s27)", "50",  "300", "b4_hybrid", "20261127"),
    ("[4B-2] 50ms  STATIC    (600g, s27)", "50",  "300", "static",    "20261127"),
    ("[4B-3] 100ms B4_HYBRID (600g, s28)", "100", "300", "b4_hybrid", "20261128"),
    ("[4B-4] 100ms STATIC    (600g, s28)", "100", "300", "static",    "20261128"),
]

for label, limit, pairs, mo, seed in suite_4b:
    cmd = [
        MATCH_BIN, "--p1", V33_P2C, "--p2", RAW_P2C,
        "--mode", "time", "--limit", limit, "--pairs", pairs,
        "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
        "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
        "--mo", mo, "--gate-mode", "wide_safe_bnd",
        "--m25", "2.5", "--m3", "1.5", "--tt-guard", TT_GUARD,
        "--dyn-gate", DYN_GATE, "--dyn-r25-k", DYN_R25_K,
        "--seed", seed, "--csv", CSV_4B,
    ]
    if USE_CAL_MPC:
        cmd.extend(["--mpc-cal-p1", CAL_V33_CSV, "--mpc-cal-p2", CAL_RAW_CSV])
    run_match_and_log(f"{label} [cal={USE_CAL_MPC}, dyn={DYN_GATE}]: WACA_G2_V33 vs RAW_NTUPLE_64", cmd, CSV_4B)

print(f"\n=== [Cell 4B Done] Full Deep CSV Summary ({CSV_4B}) ===")
!cat {CSV_4B}